# ⚡ SQL Generator · Fine-tuning Notebook

Fine-tune **Qwen 2.5 Coder 7B** on 761 k rows of text-to-SQL data using
QLoRA with Unsloth.

- **Dataset**: [`DanielRegaladoCardoso/text-to-sql-mix-v2`](https://huggingface.co/datasets/DanielRegaladoCardoso/text-to-sql-mix-v2) — 723 k train rows (10 public sources combined)
- **Base model**: [`unsloth/Qwen2.5-Coder-7B-Instruct-bnb-4bit`](https://huggingface.co/unsloth/Qwen2.5-Coder-7B-Instruct-bnb-4bit)
- **Hardware**: **Colab Pro A100 40 GB recommended** (or HF Jobs A100). T4 free technically works for a small sample but will be extremely slow.
- **Expected time (1 epoch, full data)**: ~5–8 h on A100

> [`https://github.com/DanielRegaladoUMiami/sql-agent-llmops`](https://github.com/DanielRegaladoUMiami/sql-agent-llmops)

### Tip — if you're on Colab free

Sample the dataset to ~100 k rows first (cell 7 below has a `.select(...)`).
Full data needs A100 or multi-GPU.


## 1 · Check GPU

In [ ]:
!nvidia-smi

## 2 · Install

In [ ]:
!pip install -q "unsloth @ git+https://github.com/unslothai/unsloth.git"
!pip install -q unsloth_zoo
!pip install -q --no-deps trl peft accelerate bitsandbytes datasets xformers


## 3 · HuggingFace login

In [ ]:
from huggingface_hub import login
login()


## 4 · Load Qwen 2.5 Coder 7B in 4-bit

Tries the Unsloth pre-quantized repo first; if that is unavailable or
incompatible, falls back to the official Qwen checkpoint.

In [ ]:
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LEN = 2048

PRIMARY  = "unsloth/Qwen2.5-Coder-7B-Instruct-bnb-4bit"
FALLBACK = "Qwen/Qwen2.5-Coder-7B-Instruct"

def load_with_fallback(primary, fallback, max_seq_length=MAX_SEQ_LEN):
    try:
        print(f"Trying primary: {primary}")
        return FastLanguageModel.from_pretrained(
            model_name=primary, max_seq_length=max_seq_length,
            load_in_4bit=True, dtype=None,
        )
    except Exception as e:
        print(f"⚠️  Primary failed ({type(e).__name__}). Falling back to {fallback}.")
        return FastLanguageModel.from_pretrained(
            model_name=fallback, max_seq_length=max_seq_length,
            load_in_4bit=True, dtype=None,
        )

model, tokenizer = load_with_fallback(PRIMARY, FALLBACK)


## 5 · LoRA

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r                  = 16,
    target_modules     = ["q_proj", "k_proj", "v_proj", "o_proj",
                          "gate_proj", "up_proj", "down_proj"],
    lora_alpha         = 32,
    lora_dropout       = 0,
    bias               = "none",
    use_gradient_checkpointing = "unsloth",
    random_state       = 42,
)


## 6 · Load dataset

In [ ]:
from datasets import load_dataset
ds = load_dataset("DanielRegaladoCardoso/text-to-sql-mix-v2")
print(ds)
print()
print("Sample row:")
print(ds["train"][0])


## 7 · Format as chat messages + (optional) down-sample

If you're on a T4 / 3090, keep `SAMPLE_N` small (e.g. 100 000). On A100 you can set `SAMPLE_N = None` for full data.

In [ ]:
SAMPLE_N = None   # or e.g. 100_000 for T4/3090

SYSTEM_PROMPT = (
    "You are a SQL expert. Given a SQL schema and a natural-language "
    "question, generate a correct SQL query answering the question. "
    "Return only the SQL."
)

def to_chat(row):
    user_content = (
        f"### Schema\n{row['schema_context']}\n\n"
        f"### Question\n{row['instruction']}"
    )
    messages = [
        {"role": "system",    "content": SYSTEM_PROMPT},
        {"role": "user",      "content": user_content},
        {"role": "assistant", "content": row["sql"]},
    ]
    return {"text": tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )}

train_raw = ds["train"]
if SAMPLE_N:
    train_raw = train_raw.shuffle(seed=42).select(range(SAMPLE_N))

train_ds = train_raw.map(to_chat, remove_columns=train_raw.column_names)
eval_ds  = ds["validation"].map(to_chat, remove_columns=ds["validation"].column_names)
print(train_ds)
print(train_ds[0]["text"][:1000])


## 7.5 · Filter rows that exceed MAX_SEQ_LEN

SQL schemas in this mix can be large (multi-table with FKs). A silent truncation
would make the model learn SQL without seeing the full schema. We filter
upfront — expect ~5-15% of rows to be dropped on the SQL mix.

In [ ]:
def fits(row):
    return len(tokenizer.encode(row["text"], add_special_tokens=False)) <= MAX_SEQ_LEN

_bt, _be = len(train_ds), len(eval_ds)
train_ds = train_ds.filter(fits, num_proc=4)
eval_ds  = eval_ds.filter(fits,  num_proc=4)
print(f"Train: kept {len(train_ds):,} / {_bt:,} ({100*len(train_ds)/max(1,_bt):.1f}%)")
print(f"Eval : kept {len(eval_ds):,} / {_be:,} ({100*len(eval_ds)/max(1,_be):.1f}%)")


## 8 · Trainer config

In [ ]:
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir                  = "sql_generator_adapter",
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 8,
    warmup_ratio                = 0.03,
    num_train_epochs            = 1,
    learning_rate               = 1e-4,
    fp16                        = not torch.cuda.is_bf16_supported(),
    bf16                        = torch.cuda.is_bf16_supported(),
    logging_steps               = 50,
    save_steps                  = 1000,
    save_total_limit            = 2,
    optim                       = "adamw_8bit",
    lr_scheduler_type           = "cosine",
    seed                        = 42,
    report_to                   = "none",
    dataset_text_field          = "text",
    max_seq_length              = MAX_SEQ_LEN,
    packing                     = False,
)

trainer = SFTTrainer(
    model          = model,
    tokenizer      = tokenizer,
    train_dataset  = train_ds,
    eval_dataset   = eval_ds.select(range(min(500, len(eval_ds)))),
    args           = training_args,
)


## 9 · Train 🚀

In [ ]:
trainer.train()

## 10 · Inference test

In [ ]:
FastLanguageModel.for_inference(model)

sample = ds["test"][0]
user_content = (
    f"### Schema\n{sample['schema_context']}\n\n"
    f"### Question\n{sample['instruction']}"
)
messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user",   "content": user_content},
]
input_ids = tokenizer.apply_chat_template(
    messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
).to("cuda")
out = model.generate(input_ids, max_new_tokens=400, do_sample=False, temperature=0)
print("Generated SQL:")
print(tokenizer.decode(out[0][input_ids.shape[1]:], skip_special_tokens=True))
print("\n--- Gold SQL ---")
print(sample["sql"])


## 11 · Push adapter to HuggingFace

In [ ]:
model.push_to_hub("DanielRegaladoCardoso/sql-generator-qwen25-coder-7b-lora",
                  save_method="lora")
tokenizer.push_to_hub("DanielRegaladoCardoso/sql-generator-qwen25-coder-7b-lora")


## ✅ Done

Your adapter is at:
**https://huggingface.co/DanielRegaladoCardoso/sql-generator-qwen25-coder-7b-lora**

### Optional: push a merged 16-bit model

LoRA adapters alone (~300 MB) are great for flexibility, but sometimes you
want a single deployable checkpoint:

```python
model.push_to_hub_merged(
    "DanielRegaladoCardoso/sql-generator-qwen25-coder-7b",
    tokenizer, save_method="merged_16bit"
)
```
